# Stage 11 — Lookback Comparison

**Tujuan:** Menentukan lookback terbaik (LB7 vs LB14 vs LB30) untuk klasifikasi risiko banjir menggunakan artefak Stage 10, dengan seluruh faktor lain (fitur, label, split data, arsitektur, optimizer, learning rate, batch size, callback, class weight strategy) dibuat identik. Variabel yang berbeda **hanya** panjang lookback.

**Fitur (8):** RR, Tavg, RH, CIN, KINDEX, LI, TT, SWEAT (CAPE tidak digunakan).

**Catatan penting soal metrik POD/FAR/CSI:** Dataset ini memiliki 4 kelas (multi-level flood risk). POD, FAR, dan CSI secara baku didefinisikan untuk kasus biner (event vs non-event). Pada notebook ini, ketiganya dihitung per kelas secara one-vs-rest dari confusion matrix lalu di-macro-average, sehingga tetap dapat dibandingkan secara adil antar lookback. Asumsi ini didokumentasikan ulang di `STAGE11_REPORT.md`.


## 1. Import Library

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_DIR = "."   # ganti sesuai lokasi file .npy Anda
OUTPUT_DIR = "."  # ganti sesuai lokasi output yang diinginkan

LOOKBACKS = [7, 14, 30]
FEATURES = ["RR", "Tavg", "RH", "CIN", "KINDEX", "LI", "TT", "SWEAT"]
N_CLASSES = 4
CLASS_NAMES = [f"Class {i}" for i in range(N_CLASSES)]

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))


## 2. Load Dataset

Memuat artefak Stage 10 untuk masing-masing lookback: `X_train`, `y_train`, `X_validation`, `y_validation`, `X_test`, `y_test`.


In [ ]:
def load_split_name(lb):
    """Nama file mengikuti konvensi upload: X_train_LB{lb}.npy, dst."""
    return {
        "X_train": f"X_train_LB{lb}.npy",
        "y_train": f"y_train_LB{lb}.npy",
        "X_val":   f"X_validation_LB{lb}.npy",
        "y_val":   f"y_validation_LB{lb}.npy",
        "X_test":  f"X_test_LB{lb}.npy",
        "y_test":  f"y_test_LB{lb}.npy",
    }

data = {}
for lb in LOOKBACKS:
    paths = load_split_name(lb)
    data[lb] = {k: np.load(os.path.join(DATA_DIR, v)) for k, v in paths.items()}
    print(f"LB{lb} loaded.")


## 3. Audit Dataset

Memeriksa bentuk array, jumlah fitur, tipe data, dan keberadaan NaN/Inf untuk memastikan tidak ada kebocoran data atau anomali sebelum training.

In [ ]:
audit_rows = []
for lb in LOOKBACKS:
    d = data[lb]
    for split in ["train", "val", "test"]:
        X = d[f"X_{split}"]
        y = d[f"y_{split}"]
        assert X.shape[1] == lb, f"Lookback dimensi X tidak sesuai untuk LB{lb}/{split}"
        assert X.shape[2] == len(FEATURES), f"Jumlah fitur tidak sesuai untuk LB{lb}/{split}"
        audit_rows.append({
            "lookback": lb,
            "split": split,
            "n_samples": X.shape[0],
            "timesteps": X.shape[1],
            "n_features": X.shape[2],
            "X_dtype": str(X.dtype),
            "y_dtype": str(y.dtype),
            "X_min": float(np.nanmin(X)),
            "X_max": float(np.nanmax(X)),
            "n_nan_X": int(np.isnan(X).sum()),
            "n_nan_y": int(np.isnan(y.astype(float)).sum()),
        })

audit_df = pd.DataFrame(audit_rows)
audit_df

## 4. Distribusi Kelas

Memeriksa keseimbangan kelas pada setiap split dan setiap lookback.

In [ ]:
dist_rows = []
for lb in LOOKBACKS:
    d = data[lb]
    for split in ["train", "val", "test"]:
        y = d[f"y_{split}"]
        vals, counts = np.unique(y, return_counts=True)
        row = {"lookback": lb, "split": split, "n_total": len(y)}
        for c in range(N_CLASSES):
            row[f"class_{c}"] = int(counts[vals == c][0]) if c in vals else 0
        dist_rows.append(row)

class_dist_df = pd.DataFrame(dist_rows)
class_dist_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, lb in zip(axes, LOOKBACKS):
    sub = class_dist_df[(class_dist_df.lookback == lb) & (class_dist_df.split == "train")]
    counts = [sub[f"class_{c}"].values[0] for c in range(N_CLASSES)]
    ax.bar(CLASS_NAMES, counts, color="#4C72B0")
    ax.set_title(f"LB{lb} — Distribusi Kelas (Train)")
    ax.set_ylabel("Jumlah sampel")
    for i, v in enumerate(counts):
        ax.text(i, v + max(counts) * 0.01, str(v), ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "class_distribution_train.png"), dpi=150)
plt.show()

## 5. Hitung Class Weight

Class weight dihitung **secara terpisah** dari label training masing-masing lookback (bukan digabung atau di-*hardcode*), sesuai aturan eksperimen.

In [ ]:
class_weights = {}
for lb in LOOKBACKS:
    y_train = data[lb]["y_train"]
    classes = np.unique(y_train)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
    cw_dict = {int(c): float(w) for c, w in zip(classes, weights)}
    # Pastikan semua kelas 0..N_CLASSES-1 punya entry (fallback weight=1.0 jika kelas tidak muncul di train)
    for c in range(N_CLASSES):
        cw_dict.setdefault(c, 1.0)
    class_weights[lb] = cw_dict
    print(f"LB{lb} class_weight:", cw_dict)

### Fungsi Bantu: Arsitektur Model & Metrik Evaluasi

Arsitektur, optimizer, loss, dan callback dibuat identik untuk seluruh lookback — hanya `input_shape` (ditentukan oleh panjang lookback) yang berbeda.

In [ ]:
def build_model(input_shape, n_classes=N_CLASSES):
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(n_classes, activation="softmax"),
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss=SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
    )
    return model

def get_callbacks():
    return [
        EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5),
    ]

def compute_pod_far_csi(cm):
    """Hitung POD, FAR, CSI per kelas (one-vs-rest) dari confusion matrix, lalu macro-average.
    TP = cm[c, c]; FN = sum baris c - TP; FP = sum kolom c - TP.
    """
    n_classes = cm.shape[0]
    pod_list, far_list, csi_list = [], [], []
    for c in range(n_classes):
        TP = cm[c, c]
        FN = cm[c, :].sum() - TP
        FP = cm[:, c].sum() - TP
        pod = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        far = FP / (TP + FP) if (TP + FP) > 0 else 0.0
        csi = TP / (TP + FP + FN) if (TP + FP + FN) > 0 else 0.0
        pod_list.append(pod)
        far_list.append(far)
        csi_list.append(csi)
    return {
        "POD_macro": float(np.mean(pod_list)),
        "FAR_macro": float(np.mean(far_list)),
        "CSI_macro": float(np.mean(csi_list)),
    }

def evaluate_split(model, X, y, label_names=CLASS_NAMES):
    y_pred = np.argmax(model.predict(X, verbose=0), axis=1)
    acc = accuracy_score(y, y_pred)
    prec = precision_score(y, y_pred, average="macro", zero_division=0)
    rec = recall_score(y, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y, y_pred, labels=list(range(len(label_names))))
    pfc = compute_pod_far_csi(cm)
    metrics = {
        "Accuracy": acc,
        "Precision_Macro": prec,
        "Recall_Macro": rec,
        "F1_Macro": f1,
        **pfc,
    }
    return metrics, cm, y_pred

## 6. Training LB7

In [ ]:
lb = 7
X_train, y_train = data[lb]["X_train"], data[lb]["y_train"]
X_val, y_val = data[lb]["X_val"], data[lb]["y_val"]

model_lb7 = build_model(input_shape=(X_train.shape[1], X_train.shape[2]))
model_lb7.summary()

history_lb7 = model_lb7.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    class_weight=class_weights[lb],
    callbacks=get_callbacks(),
    verbose=1,
)

history_df_lb7 = pd.DataFrame(history_lb7.history)
history_df_lb7.insert(0, "epoch", range(1, len(history_df_lb7) + 1))
history_df_lb7.to_csv(os.path.join(OUTPUT_DIR, "training_history_lb7.csv"), index=False)
history_df_lb7.tail()

## 7. Training LB14

In [ ]:
lb = 14
X_train, y_train = data[lb]["X_train"], data[lb]["y_train"]
X_val, y_val = data[lb]["X_val"], data[lb]["y_val"]

model_lb14 = build_model(input_shape=(X_train.shape[1], X_train.shape[2]))
model_lb14.summary()

history_lb14 = model_lb14.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    class_weight=class_weights[lb],
    callbacks=get_callbacks(),
    verbose=1,
)

history_df_lb14 = pd.DataFrame(history_lb14.history)
history_df_lb14.insert(0, "epoch", range(1, len(history_df_lb14) + 1))
history_df_lb14.to_csv(os.path.join(OUTPUT_DIR, "training_history_lb14.csv"), index=False)
history_df_lb14.tail()

## 8. Training LB30

In [ ]:
lb = 30
X_train, y_train = data[lb]["X_train"], data[lb]["y_train"]
X_val, y_val = data[lb]["X_val"], data[lb]["y_val"]

model_lb30 = build_model(input_shape=(X_train.shape[1], X_train.shape[2]))
model_lb30.summary()

history_lb30 = model_lb30.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    class_weight=class_weights[lb],
    callbacks=get_callbacks(),
    verbose=1,
)

history_df_lb30 = pd.DataFrame(history_lb30.history)
history_df_lb30.insert(0, "epoch", range(1, len(history_df_lb30) + 1))
history_df_lb30.to_csv(os.path.join(OUTPUT_DIR, "training_history_lb30.csv"), index=False)
history_df_lb30.tail()

## 9. Evaluasi Validation

In [ ]:
val_metrics = {}
val_preds = {}
val_cms = {}

for lb, model in zip(LOOKBACKS, [model_lb7, model_lb14, model_lb30]):
    X_val, y_val = data[lb]["X_val"], data[lb]["y_val"]
    metrics, cm, y_pred = evaluate_split(model, X_val, y_val)
    val_metrics[lb] = metrics
    val_preds[lb] = y_pred
    val_cms[lb] = cm
    print(f"LB{lb} (Validation):", metrics)

## 10. Evaluasi Test

In [ ]:
test_metrics = {}
test_preds = {}
test_cms = {}

for lb, model in zip(LOOKBACKS, [model_lb7, model_lb14, model_lb30]):
    X_test, y_test = data[lb]["X_test"], data[lb]["y_test"]
    metrics, cm, y_pred = evaluate_split(model, X_test, y_test)
    test_metrics[lb] = metrics
    test_preds[lb] = y_pred
    test_cms[lb] = cm
    print(f"LB{lb} (Test):", metrics)

In [ ]:
for lb in LOOKBACKS:
    rows = []
    for split_name, m in [("validation", val_metrics[lb]), ("test", test_metrics[lb])]:
        row = {"lookback": lb, "split": split_name}
        row.update(m)
        rows.append(row)
    pd.DataFrame(rows).to_csv(os.path.join(OUTPUT_DIR, f"metrics_lb{lb}.csv"), index=False)

pd.concat([
    pd.DataFrame([{"lookback": lb, "split": "validation", **val_metrics[lb]} for lb in LOOKBACKS]),
    pd.DataFrame([{"lookback": lb, "split": "test", **test_metrics[lb]} for lb in LOOKBACKS]),
]).reset_index(drop=True)

## 11. Confusion Matrix

In [ ]:
def save_cm_csv(cm, path):
    cm_df = pd.DataFrame(cm, index=[f"true_{c}" for c in range(N_CLASSES)],
                              columns=[f"pred_{c}" for c in range(N_CLASSES)])
    cm_df.to_csv(path)
    return cm_df

for lb in LOOKBACKS:
    save_cm_csv(val_cms[lb], os.path.join(OUTPUT_DIR, f"confusion_matrix_lb{lb}_validation.csv"))
    save_cm_csv(test_cms[lb], os.path.join(OUTPUT_DIR, f"confusion_matrix_lb{lb}_test.csv"))

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for i, lb in enumerate(LOOKBACKS):
    sns.heatmap(val_cms[lb], annot=True, fmt="d", cmap="Blues", ax=axes[0, i],
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    axes[0, i].set_title(f"LB{lb} — Validation")
    axes[0, i].set_xlabel("Predicted"); axes[0, i].set_ylabel("True")

    sns.heatmap(test_cms[lb], annot=True, fmt="d", cmap="Oranges", ax=axes[1, i],
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    axes[1, i].set_title(f"LB{lb} — Test")
    axes[1, i].set_xlabel("Predicted"); axes[1, i].set_ylabel("True")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrices_all.png"), dpi=150)
plt.show()

## 12. Perbandingan Hasil

In [ ]:
comparison_rows = []
for lb in LOOKBACKS:
    row = {"lookback": f"LB{lb}"}
    row.update({f"val_{k}": v for k, v in val_metrics[lb].items()})
    row.update({f"test_{k}": v for k, v in test_metrics[lb].items()})
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(os.path.join(OUTPUT_DIR, "comparison_results.csv"), index=False)
comparison_df

## 13. Pemilihan Lookback Terbaik

Urutan prioritas pemilihan (bukan berdasarkan Accuracy tertinggi semata):

1. F1 Macro (Test)
2. Recall Macro (Test)
3. POD (Test)
4. CSI (Test)
5. Accuracy (Test)

Evaluasi pada **Test set** digunakan sebagai basis keputusan akhir karena mencerminkan performa generalisasi model.


In [ ]:
priority_cols = ["test_F1_Macro", "test_Recall_Macro", "test_POD_macro", "test_CSI_macro", "test_Accuracy"]

ranking_df = comparison_df.sort_values(by=priority_cols, ascending=False).reset_index(drop=True)
ranking_df.insert(0, "rank", range(1, len(ranking_df) + 1))

best_lookback = ranking_df.loc[0, "lookback"]

print("Ranking lookback (berdasarkan prioritas F1 Macro > Recall Macro > POD > CSI > Accuracy, pada Test set):")
display_cols = ["rank", "lookback"] + priority_cols
print(ranking_df[display_cols].to_string(index=False))
print()
print(f"Lookback terbaik: {best_lookback}")

## 14. Simpan Model

In [ ]:
model_lb7.save(os.path.join(OUTPUT_DIR, "model_lb7.keras"))
model_lb14.save(os.path.join(OUTPUT_DIR, "model_lb14.keras"))
model_lb30.save(os.path.join(OUTPUT_DIR, "model_lb30.keras"))

saved_files = [
    "model_lb7.keras", "model_lb14.keras", "model_lb30.keras",
    "training_history_lb7.csv", "training_history_lb14.csv", "training_history_lb30.csv",
    "confusion_matrix_lb7_validation.csv", "confusion_matrix_lb14_validation.csv", "confusion_matrix_lb30_validation.csv",
    "confusion_matrix_lb7_test.csv", "confusion_matrix_lb14_test.csv", "confusion_matrix_lb30_test.csv",
    "metrics_lb7.csv", "metrics_lb14.csv", "metrics_lb30.csv",
    "comparison_results.csv",
]
for f in saved_files:
    path = os.path.join(OUTPUT_DIR, f)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"[{status}] {f}")

## 15. Kesimpulan

In [ ]:
print("="*60)
print("RINGKASAN EKSPERIMEN STAGE 11 — LOOKBACK COMPARISON")
print("="*60)
print(ranking_df[["rank", "lookback"] + priority_cols].to_string(index=False))
print()
print(f"Lookback terbaik: {best_lookback}")
print()
print("Detail metrik Test set:")
for lb in LOOKBACKS:
    print(f"  LB{lb}: {test_metrics[lb]}")
print()
print("Seluruh artefak (model, history, confusion matrix, metrics, comparison_results.csv)")
print("telah disimpan di:", os.path.abspath(OUTPUT_DIR))
print()
print("Langkah selanjutnya: lengkapi bagian Analisis & Kesimpulan di STAGE11_REPORT.md")
print("dengan angka-angka aktual di atas, lalu tinjau confusion matrix untuk interpretasi kualitatif.")